Phase A ? univers officiel de **51 entreprises**. Code corrig?, non r?ex?cut?.
Les sorties ant?rieures ont ?t? effac?es pour ?viter de les attribuer aux nouvelles r?gles.
Signal ? 18:00 MYT ; publication apr?s 18:00 ? prochain signal r?ellement disponible. Targets inchang?es. Rebuild uniquement apr?s autorisation de la Phase B.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from finagent.forecast.alignment import align_news_to_signals, publication_timestamp, MARKET_TIMEZONE

In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================

market_df = pd.read_csv("../data/preprocessed_data/malaysia_market_features.csv")
news_df = pd.read_csv("../data/preprocessed_data/news_data_sentiment_rag.csv")
print("Market shape :", market_df.shape)
print("News shape   :", news_df.shape)
display(market_df.head())
display(news_df.head())

In [ ]:
print("MARKET COLUMNS")
print(
    market_df.columns.tolist()
)


print("\nNEWS COLUMNS")
print(
    news_df.columns.tolist()
)

In [ ]:
market_df["date"] = pd.to_datetime(market_df["date"]).dt.normalize()
# Preserve every original news row: alignment records why older or late news is excluded.
news_df["publication_time_my"] = news_df.apply(publication_timestamp, axis=1)
print("Available market period:", market_df["date"].min(), "to", market_df["date"].max())
print("Daily signal: 18:00 Asia/Kuala_Lumpur; target: next-session return.")


In [ ]:
# The signal calendar is built per ticker from observed market sessions.

In [ ]:
print(
    "Market tickers:",
    sorted(
        market_df["ticker"]
        .dropna()
        .unique()
    )
)


print(
    "\nNews tickers:",
    sorted(
        news_df["ticker"]
        .dropna()
        .unique()
    )
)

In [ ]:
# 18:00 MYT is the exact signal cutoff, including publications at 18:00.
# Weekends/holidays and after-cutoff publications map to the next actual signal.

In [ ]:
# Shared, tested implementation. It rejects date-only news and retains the
# original timestamps plus publication_time_my, signal_time_my and mapping_status.
from finagent.forecast.alignment import align_news_to_signals

In [ ]:
news_df = align_news_to_signals(news_df, market_df)
print(news_df["mapping_status"].value_counts())
valid_mapping = news_df["signal_time_my"].notna()
assert (news_df.loc[valid_mapping, "publication_time_my"] <=
        news_df.loc[valid_mapping, "signal_time_my"]).all()

In [ ]:
display(news_df[["ticker", "date", "datetime_my", "publication_time_my",
                 "market_date", "signal_time_my", "mapping_status", "sentiment_score"]].head(20))

In [ ]:
news_df["is_positive"] = (
    news_df["sentiment"]
    .str.lower()
    .eq("positive")
    .astype(int)
)


news_df["is_negative"] = (
    news_df["sentiment"]
    .str.lower()
    .eq("negative")
    .astype(int)
)


news_df["is_neutral"] = (
    news_df["sentiment"]
    .str.lower()
    .eq("neutral")
    .astype(int)
)

In [ ]:
# ============================================================
# 4. DAILY NEWS AGGREGATION
# ============================================================

daily_news = (
    news_df
    .dropna(
        subset=["market_date"]
    )
    .groupby(
        [
            "ticker",
            "market_date"
        ],
        as_index=False
    )
    .agg(
        news_count=(
            "sentiment_score",
            "size"
        ),

        sentiment_mean=(
            "sentiment_score",
            "mean"
        ),

        sentiment_max=(
            "sentiment_score",
            "max"
        ),

        sentiment_min=(
            "sentiment_score",
            "min"
        ),

        positive_score_mean=(
            "positive_score",
            "mean"
        ),

        negative_score_mean=(
            "negative_score",
            "mean"
        ),

        neutral_score_mean=(
            "neutral_score",
            "mean"
        ),

        positive_count=(
            "is_positive",
            "sum"
        ),

        negative_count=(
            "is_negative",
            "sum"
        ),

        neutral_count=(
            "is_neutral",
            "sum"
        )
    )
)

In [ ]:
display(
    daily_news.head(20)
)

print(
    "Daily news shape:",
    daily_news.shape
)

In [ ]:
# ============================================================
# 5. MERGE MARKET + NEWS
# ============================================================

forecast_df = market_df.merge(
    daily_news,

    left_on=[
        "ticker",
        "date"
    ],

    right_on=[
        "ticker",
        "market_date"
    ],

    how="left"
)

In [ ]:
forecast_df = forecast_df.drop(
    columns=[
        "market_date"
    ]
)

In [ ]:
news_features = [
    "news_count",
    "sentiment_mean",
    "sentiment_max",
    "sentiment_min",
    "positive_score_mean",
    "negative_score_mean",
    "neutral_score_mean",
    "positive_count",
    "negative_count",
    "neutral_count"
]


forecast_df[
    news_features
] = (
    forecast_df[
        news_features
    ]
    .fillna(0)
)

In [ ]:
display(
    forecast_df[
        [
            "date",
            "company",
            "ticker",
            "close",
            "daily_return",
            "rsi_14",
            "macd",
            "news_count",
            "sentiment_mean",
            "positive_count",
            "negative_count"
        ]
    ].tail(30)
)

In [ ]:
print(
    forecast_df.shape
)


print(
    forecast_df[
        news_features
    ].isnull().sum()
)

In [ ]:
news_days = (
    forecast_df[
        "news_count"
    ] > 0
)


print(
    "Trading days:",
    len(forecast_df)
)


print(
    "Trading days with news:",
    news_days.sum()
)


print(
    "Percentage with news:",
    round(
        news_days.mean() * 100,
        2
    ),
    "%"
)

In [ ]:
forecast_df = forecast_df.sort_values(
    [
        "ticker",
        "date"
    ]
).reset_index(
    drop=True
)

In [ ]:
forecast_df[
    "target_return_1d"
] = (
    forecast_df
    .groupby("ticker")[
        "daily_return"
    ]
    .shift(-1)
)

In [ ]:
forecast_df[
    "target_direction_1d"
] = np.where(
    forecast_df[
        "target_return_1d"
    ].isna(),

    np.nan,

    np.where(
        forecast_df[
            "target_return_1d"
        ] > 0,
        1,
        0
    )
)

In [ ]:
forecast_df[
    "target_return_5d"
] = (
    forecast_df
    .groupby(
        "ticker"
    )["close"]
    .transform(
        lambda x:
            (
                x.shift(-5)
                /
                x
                -
                1
            )
            * 100
    )
)

In [ ]:
display(
    forecast_df[
        [
            "date",
            "company",
            "close",
            "daily_return",
            "target_return_1d",
            "target_direction_1d",
            "target_return_5d"
        ]
    ].tail(15)
)

In [ ]:
important_columns = [
    "close",
    "volume",
    "daily_return",
    "rsi_14",
    "macd",
    "atr_14",
    "volatility_20",
    "momentum_10",
    "news_count",
    "sentiment_mean",
    "target_return_1d"
]


print(
    forecast_df[
        important_columns
    ].isnull().sum()
)

In [ ]:
output_dir = Path("../data/preprocessed_data")
output_dir.mkdir(parents=True, exist_ok=True)
forecast_df.to_csv(output_dir / "forecasting_dataset.csv", index=False)
# Saved only when Phase B explicitly executes the notebook: auditable origins
# and reasons for every retained/rejected publication in the complete source.
news_df.to_csv(output_dir / "news_forecast_alignment.csv", index=False)
print("Forecast dataset and news mapping audit saved.")

In [ ]:
news_summary = (
    forecast_df
    .groupby(
        [
            "company",
            "ticker"
        ]
    )
    .apply(
        lambda group: pd.Series(
            {
                "trading_days":
                    len(group),

                "days_with_news":
                    (
                        group["news_count"] > 0
                    ).sum(),

                "total_news":
                    group["news_count"].sum(),

                "sentiment_mean_on_news_days":
                    group.loc[
                        group["news_count"] > 0,
                        "sentiment_mean"
                    ].mean()
            }
        ),
        include_groups=False
    )
    .reset_index()
)

In [ ]:
print(
    "Market:",
    market_df["date"].min(),
    "→",
    market_df["date"].max()
)

print(
    "News:",
    news_df["date"].min(),
    "→",
    news_df["date"].max()
)

display(news_summary)

display(
    forecast_df[
        forecast_df["news_count"] > 0
    ].head(20)
)

In [ ]:
feature_columns = [
    "open", "high", "low", "close", "volume",
    "daily_return", "intraday_return", "daily_range", "volume_change",

    "rsi_14",
    "sma_20", "sma_50", "sma_200",
    "ema_12", "ema_26",
    "macd", "macd_signal", "macd_histogram",
    "atr_14",
    "bollinger_width",
    "volatility_20",
    "momentum_10",

    "news_count",
    "sentiment_mean",
    "sentiment_max",
    "sentiment_min",
    "positive_score_mean",
    "negative_score_mean",
    "neutral_score_mean",
    "positive_count",
    "negative_count",
    "neutral_count"
]

In [ ]:
model_df = (
    forecast_df
    .dropna(
        subset=feature_columns + ["target_return_1d"]
    )
    .sort_values(
        ["ticker", "date"]
    )
    .reset_index(drop=True)
    .copy()
)


print(
    "Original forecast dataset:",
    forecast_df.shape
)


print(
    "Model dataset:",
    model_df.shape
)


print(
    "\nPeriod:"
)

print(
    model_df["date"].min(),
    "→",
    model_df["date"].max()
)


print(
    "\nRows per company:"
)

display(
    model_df[
        "company"
    ].value_counts()
)

model_df.to_csv(output_dir / "forecasting_model_dataset.csv",index=False)

print(
    "Model dataset saved."
)